## Installing Packages

In [1]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Importing Libraries

In [2]:
import os
import json # JSON Tool Schema
from openai import AzureOpenAI # OpenAI API client

from datetime import datetime
from zoneinfo import ZoneInfo

# Define the deployment you want to use for your chat completions API calls

In [3]:
deployment_name = "mindcraft-gpt4o"

In [4]:
# Initialize Azure OpenAI client

client = AzureOpenAI(
    api_version="2024-02-01", # Use a stable API version
    azure_endpoint="https://mindcraft-kapidhwaj.openai.azure.com/", # Fixed URL - removed the incorrect path
    api_key="8WxLaoodYxa7XSK2rCiWuP3nqwWUShSUVd5FrjEYSqqROfIwc0qzJQQJ99BFAC77bzfXJ3w3AAABACOGweqQ"
)

In [5]:
# Test the connection
try:
    # Test with a simple completion to verify the connection works
    test_response = client.chat.completions.create(
        model=deployment_name,
        messages=[{"role": "user", "content": "Hello"}],
        max_tokens=10
    )
    print("✅ Azure OpenAI connection successful!")
    print(f"Test response: {test_response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("Please check:")
    print("1. Your deployment name is correct")
    print("2. Your endpoint URL is correct") 
    print("3. Your API key is valid and not expired")
    print("4. The deployment is active in Azure")

❌ Connection failed: Connection error.
Please check:
1. Your deployment name is correct
2. Your endpoint URL is correct
3. Your API key is valid and not expired
4. The deployment is active in Azure


## Simplified timezone data

In [6]:
TIMEZONE_DATA = {
    "tokyo": "Asia/Tokyo",
    "san francisco": "America/Los_Angeles",
    "paris": "Europe/Paris"
}

## Creating Function to retrieve Current Time

In [7]:
def get_current_time(location):
    """Get the current time for a given location"""
    print(f"get_current_time called with location: {location}")  # take the location
    location_lower = location.lower() # lowercase the location for case-insensitive matching
    
    for key, timezone in TIMEZONE_DATA.items():
        if key in location_lower:
            print(f"Timezone found for {key}")  
            current_time = datetime.now(ZoneInfo(timezone)).strftime("%I:%M %p")
            return json.dumps({
                "location": location,
                "current_time": current_time
            })
    
    print(f"No timezone data found for {location_lower}")  
    return json.dumps({"location": location, "current_time": "unknown"})

## Creating Function to retrieve favorite color

In [8]:
def favorite_color():
    """Return Ishaan's favorite color"""
    print("favorite_color called")  # Log the function call
    return json.dumps({"color": "red"})  # Return Ishaan's favorite color in JSON format

## Creating Function for conversation

In [9]:
def run_conversation():
    try:
        # Initial user message
        messages = [{"role": "user", "content": "What's the current time in San Francisco"}] # Single function call
        #messages = [{"role": "user", "content": "What's the current time in San Francisco, Tokyo, and Paris?"}] # Parallel function call with a single tool/function defined

        # Define the function for the model
        tools = [
            {
                "type": "function",
                "function": {
                    "name": "get_current_time",
                    "description": "Get the current time in a given location",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city name, e.g. San Francisco",
                            },
                        },
                        "required": ["location"],
                    },
                }
            },  # Added missing comma here
            {
                "type": "function",
                "function": {
                    "name": "favorite_color",
                    "description": "Get Ishaan's favorite color",
                    "parameters": {
                        "type": "object",
                        "properties": {},  # No parameters needed for this function
                        "required": [],
                    },
                }
            }
        ]

        print("🔄 Making first API call...")
        # First API call: Ask the model to use the function
        response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        # Process the model's response
        response_message = response.choices[0].message
        messages.append(response_message)

        print("📨 Model's response:")  
        print(response_message)  

        # Handle function calls
        if response_message.tool_calls:
            print(f"🔧 Model wants to call {len(response_message.tool_calls)} function(s)")
            for tool_call in response_message.tool_calls:
                if tool_call.function.name == "get_current_time":
                    function_args = json.loads(tool_call.function.arguments)
                    print(f"📍 Getting time for: {function_args}")  
                    time_response = get_current_time(
                        location=function_args.get("location")
                    )
                    messages.append({
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": "get_current_time",
                        "content": time_response,
                    })
                elif tool_call.function.name == "favorite_color":
                    print("🎨 Getting favorite color...")
                    color_response = favorite_color()
                    messages.append({
                        "tool_call_id": tool_call.id,
                        "role": "tool", 
                        "name": "favorite_color",
                        "content": color_response,
                    })
        else:
            print("❌ No tool calls were made by the model.")  

        print("🔄 Making final API call...")
        # Second API call: Get the final response from the model
        final_response = client.chat.completions.create(
            model=deployment_name,
            messages=messages,
        )

        return final_response.choices[0].message.content
    
    except Exception as e:
        print(f"❌ Error in run_conversation: {e}")
        return f"Error occurred: {str(e)}"

## Running the conversation

In [10]:
# Run the conversation and print the result
print(run_conversation())

🔄 Making first API call...
❌ Error in run_conversation: Connection error.
Error occurred: Connection error.
❌ Error in run_conversation: Connection error.
Error occurred: Connection error.
